[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/karzit/temp/blob/master/notebooks/ml-curriculum/01_basic_classification/01_basic_classification.ipynb)

# ML 튜토리얼: 기본 분류 모델 (Iris 데이터셋)

머신러닝 입문자를 위한 튜토리얼입니다. 데이터 로드 → 탐색(EDA) → 전처리 → 모델 학습 → 평가 → 저장까지 전체 파이프라인을 다뢈니다.

## Colab에서 열기
1. 이 노트북 파일을 [colab.research.google.com](https://colab.research.google.com) → **파일 → 노트북 업로드**로 업로드하거나,
2. GitHub에 이 프로젝트를 push한 뒤 Colab의 **GitHub 탭**에서 저장소 URL로 불러오면 됩니다.

아래 첫 셀을 실행하면 Colab인지 로컬인지를 자동 감지해 필요한 패키지를 설치하고, 원한다면 Google Drive도 마운트합니다.

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules
print("Running in Colab:", IN_COLAB)

if IN_COLAB:
    !pip install -q scikit-learn pandas matplotlib seaborn joblib

    # 필요 시 주석 해제해서 Google Drive 마운트 (모델/데이터 저장용)
    # from google.colab import drive
    # drive.mount('/content/drive')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import joblib

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42

## 1. 데이터 로드

In [ ]:
iris = load_iris(as_frame=True)
df = iris.frame.copy()
df["species"] = df["target"].map(dict(enumerate(iris.target_names)))
df.head()

## 2. 탐색적 데이터 분석 (EDA)

In [ ]:
df.describe()

In [ ]:
sns.pairplot(df, hue="species", vars=iris.feature_names)
plt.show()

## 3. 학습/테스트 분리 및 전처리

In [ ]:
X = df[iris.feature_names]
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train.shape, X_test.shape

## 4. 모델 학습

In [ ]:
model = LogisticRegression(max_iter=200, random_state=RANDOM_STATE)
model.fit(X_train_scaled, y_train)

## 5. 평가

In [ ]:
y_pred = model.predict(X_test_scaled)

print("Accuracy:", accuracy_score(y_test, y_pred))
print()
print(classification_report(y_test, y_pred, target_names=iris.target_names))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt="d", xticklabels=iris.target_names, yticklabels=iris.target_names, cmap="Blues")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()

## 6. 모델 저장

Colab에서는 세션이 끝나면 `/content`가 사라지므로, 계속 보관하려면 위에서 Drive를 마운트하고 경로를 조정하세요.

In [ ]:
import os
os.makedirs("../../../models", exist_ok=True)
joblib.dump(model, "../../../models/iris_logreg.joblib")
joblib.dump(scaler, "../../../models/iris_scaler.joblib")
print("Saved.")

## 다음 단계 아이디어
- 다른 모델(RandomForest, SVM 등)과 성능 비교해보기
- `GridSearchCV`로 하이퍼파라미터 튜닝해보기
- 자신만의 CSV 데이터셋을 `data/` 폴더에 넣고 동일한 파이프라인 적용해보기
- 이미지/텍스트 데이터로 넘어가고 싶다면 PyTorch 기반 튜토리얼로 확장 가능